# Microbots: a GNN + GRU policy that knows when it's needed, and when to go home

With 3× or 5× more bots than a shape needs, two things should happen:

- **(a) Only the bots that are needed go to work.** The rest stay where they are.
- **(b) When the shape is finished, the spares go home** to the pad where they started, instead of sitting in a heap on top of the structure.

Neither can be decided by one bot alone: whether you're one of the bots nearest the work, and whether the shape is finished out of sight, are facts about the whole swarm. The hand-written rules (the **teacher**) cheat and look at the whole grid. The network has to work it out from its neighbours:

- **GNN:** every step, each bot sends messages to the bots touching it and combines what it hears (3 rounds per step).
- **GRU:** each bot carries a memory from step to step, so news like "the work is that way" or "everything's built, go home" can keep travelling.

Each bot sees its own 5×5×5 block, whether it's locked in or parked, the relayed "how far to the work" signal bots pass to each other, and (by default) a compass pointing at the nearest open cell, or home once the shape is built. What it still has to work out from its neighbours: **am I one of the bots that's needed**, and **is the build finished**. Every bot runs the same network. It learns by copying the teacher, then DAgger (the network drives, the teacher labels).

A first try with no signal and no compass (fewer inputs, more to learn) failed: driving on its own, the swarm stalled. See `linking/EXPERIMENTS.md`, section 9.

Code: [`linking/learn/gnn_gru.py`](https://github.com/ItsAdarsh008/microbots/blob/main/linking/learn/gnn_gru.py) · the teacher's rules: `env.plan()` in [`env.py`](https://github.com/ItsAdarsh008/microbots/blob/main/linking/learn/env.py).

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Recording and testing run the simulation on the CPU either way (a few seconds per build); the GPU is for training.

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU"
!git clone -q --depth 1 https://github.com/ItsAdarsh008/microbots
%cd microbots/linking/learn
!pip -q install scipy
import jax
print("JAX", jax.__version__, "on", jax.devices())

## 1. Settings

In [ ]:
import time, numpy as np, jax, jax.numpy as jnp
import gnn_gru as G

MULT = 3          # times the usual number of bots (3 or 5)
SEEDS = 12        # teacher builds per shape (wall and arch)
EPOCHS = 40
MEMORY = True     # False = wipe each bot's memory every step, for comparison
G.USE_SIGNAL = True     # the relayed "how far to the work" hop count, passed bot to bot (local)
G.USE_COMPASS = True    # direction to the nearest open cell / home: a shortcut, but it makes training much easier
print("bots padded to", G.configure(MULT))

## 2. Record the teacher

Each build: the teacher builds the **wall** (`sheet`) or the **arch**, with only the bots nearest the work moving, then sends the spares home. At every step we save, for every bot: what it sees, who it can hear (the bots in the 26 cells touching it), its legal moves, and the teacher's move ("stay" for bots that aren't needed).

In [ ]:
builds = []
t0 = time.time()
for shape in ["sheet", "arch"]:
    for seed in range(SEEDS):
        rec, st = G.rollout(shape, seed, MULT)
        builds.append(rec)
        print(f"{shape:5s} seed {seed:2d}: built in {st['built_in']}, {st['spares_that_moved']}/{st['spares']} spares moved, home in {st['home_in']}")
print(f"{len(builds)} builds in {time.time() - t0:.0f}s")

## 3. The network

Encoder (252 inputs → 64) → 3 rounds of message passing between touching bots → GRU memory → 19 move scores (stay, 6 slides, 12 flips). Illegal moves are masked out, and a safety check underneath means no move can ever drop a bot, whatever the network picks.

In [ ]:
params = G.init(jax.random.PRNGKey(0))
opt = G.adam_init(params)
log = {"loss": [], "epoch_end": [], "agree": [], "stage": []}
print(f"{G.count(params):,} weights")

## 4. Train by imitation

4 builds at a time, 32 steps at a time, with each bot's memory carried through (truncated backpropagation through time). The last number on each line is how often it correctly stays put when the teacher would: that's behaviour (a).

In [ ]:
t0 = time.time()
params, opt = G.train(params, opt, builds, EPOCHS, MEMORY, log)
print(f"{time.time() - t0:.0f}s")

## 5. DAgger: let the network drive

In [ ]:
for r in range(2):
    for shape in ["sheet", "arch"]:
        for seed in range(1000 + 10 * r, 1000 + 10 * r + SEEDS // 2):
            rec, st = G.rollout(shape, seed, MULT, G.Driver(params, MEMORY, seed))
            builds.append(rec)
            print(f"round {r + 1} {shape:5s} seed {seed}: built in {st['built_in']}, home in {st['home_in']}")
    params, opt = G.train(params, opt, builds, EPOCHS // 4, MEMORY, log, stage=f"dagger {r + 1}")

## 6. The training curve

In [ ]:
import matplotlib.pyplot as plt
loss = np.array(log["loss"])
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(loss, color="#a9c1df", lw=0.6)
ax.plot(np.convolve(loss, np.ones(20) / 20, mode="valid"), color="#2a5ea8", lw=2)
ax.set_xlabel("training step"); ax.set_ylabel("loss")
ax2 = ax.twinx(); ax2.plot(np.array(log["epoch_end"]), np.array(log["agree"]) * 100, "o-", color="#d9622b")
ax2.set_ylabel("% same move as the teacher", color="#d9622b")
plt.title(f"GNN + GRU, {MULT}x bots"); plt.show()

## 7. Test it against the teacher

Unseen seeds; the **tower** was never trained on. What to look at:

- **built in:** steps to finish the shape (`None` = didn't finish).
- **spares moved:** of the spares, how many ever moved. Lower is better: that's (a).
- **home in / not home:** steps until every spare was back on the pad, and how many weren't. That's (b).

In [ ]:
import pandas as pd
rows = G.evaluate(params, MEMORY, MULT, seeds=range(100, 103))
rows += G.evaluate(params, MEMORY, MULT, seeds=range(100, 103), who="teacher")
df = pd.DataFrame(rows)
df.groupby(["who", "shape"])[["built_in", "spares_that_moved", "spares", "home_in", "not_home"]].mean().round(1)

## 8. Try it without memory

Set `MEMORY = False` in section 1 and run everything again. If the bots really use their memory to pass on "is the work over there" and "is everything built", this version should be worse at (a) and especially (b).

## 9. Keep the weights

In [ ]:
import pickle
name = f"gnn_{'gru' if MEMORY else 'nomem'}_x{MULT}.pkl"
pickle.dump({"params": jax.tree_util.tree_map(np.asarray, params), "memory": MEMORY, "mult": MULT,
             "signal": G.USE_SIGNAL, "compass": G.USE_COMPASS}, open(name, "wb"))
try:
    from google.colab import files; files.download(name)
except ImportError:
    print("saved", name)